# 01. Chuẩn bị thư mục dự án và mã nguồn đối chiếu

Sổ tay này chạy một lần trước khi làm thực nghiệm. Ba việc được thực hiện: dựng khung thư mục trên Google Drive; đưa mã nguồn công khai của GraFITi vào thư mục con `Code`; đối chiếu phần triển khai DLinear và NLinear của nhóm tác giả với phần triển khai của bộ code này.

Khung thư mục đích:

```
AIO2026_Conquer_M4_DLinear_NLinear_Slot4/
  Code/      ma nguon tham chieu, gom ca GraFITi
  Data/      du lieu tho tai ve tu PhysioNet
  Output/    moi ket qua, hinh, nhat ky, checkpoint
```

Việc đối chiếu mã nguồn không nhằm chạy lại GraFITi trên Colab miễn phí. Mô hình đó cần môi trường cũ và thời gian huấn luyện lớn. Mục đích là lấy đúng định nghĩa hai đường cơ sở tuyến tính mà bài báo gọi là DLinear+ và NLinear+, rồi chứng minh bằng số rằng mã của chúng ta tái tạo đúng chúng.

> **Vị trí trong chuỗi tái lập:** notebook 01 trên tổng số 31. Thứ tự chạy và sơ đồ phụ thuộc nằm trong `README.md`.
>
> **Cần chạy trước:** không có.
>
> **Thư mục kết quả:** theo mô tả trong notebook.
>
> **Mã định danh nội bộ:** `W2_00`. Mã này còn xuất hiện trong tên khóa JSON, tên hình và nhật ký chạy; nó được giữ nguyên để không phá phép đối chiếu chéo và mã băm của bản khai báo trước.

In [1]:
#@title Tham số { display-mode: "form" }
PROJECT_DIR = "/content/drive/MyDrive/AIO2026_Conquer_M4_DLinear_NLinear_Slot4"  #@param {type:"string"}
GITHUB_URL = "https://github.com/yalavarthivk/GraFITi"  #@param {type:"string"}
GIU_LAI_DU_LIEU_MAU = True  #@param {type:"boolean"}

# Thư mục data của kho mã nặng khoảng 1,2 GB và không cần cho công việc tuần 2.
# Đặt GIU_LAI_DU_LIEU_MAU bằng True nếu muốn giữ, nhưng hãy kiểm tra dung lượng Drive trước.

import os, sys, json, shutil, subprocess, datetime
from pathlib import Path

if "google.colab" in sys.modules:
    from google.colab import drive
    if not os.path.ismount("/content/drive"):
        drive.mount("/content/drive")
else:
    PROJECT_DIR = os.environ.get("P12_PROJECT", "./AIO2026_Conquer_M4_DLinear_NLinear_Slot4")

PROJECT = Path(PROJECT_DIR)
CODE = PROJECT / "Code"
DATA_RAW = PROJECT / "Data"
OUT = PROJECT / "Output"
for d in (CODE, DATA_RAW, OUT, OUT / "processed", OUT / "runs", OUT / "figures", OUT / "logs"):
    d.mkdir(parents=True, exist_ok=True)

for d in (PROJECT, CODE, DATA_RAW, OUT):
    print("đã sẵn sàng:", d)

Mounted at /content/drive
đã sẵn sàng: /content/drive/MyDrive/AIO2026_Conquer_M4_DLinear_NLinear_Slot4
đã sẵn sàng: /content/drive/MyDrive/AIO2026_Conquer_M4_DLinear_NLinear_Slot4/Code
đã sẵn sàng: /content/drive/MyDrive/AIO2026_Conquer_M4_DLinear_NLinear_Slot4/Data
đã sẵn sàng: /content/drive/MyDrive/AIO2026_Conquer_M4_DLinear_NLinear_Slot4/Output


## 1. Đưa mã nguồn GraFITi vào thư mục Code

Ô dưới đây ưu tiên sao chép kho mã từ GitHub. Nếu máy chủ không truy cập được, hãy tự tải tệp nén của kho mã lên `Code/GraFITi-main.zip` rồi chạy lại ô này: mã sẽ tự giải nén.

In [2]:
REPO = CODE / "GraFITi"
zip_path = CODE / "GraFITi-main.zip"

if REPO.exists() and (REPO / "train_grafiti.py").exists():
    print("Kho mã đã có sẵn tại", REPO)
elif zip_path.exists():
    import zipfile
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(CODE)
    cand = [p for p in CODE.iterdir() if p.is_dir() and (p / "train_grafiti.py").exists()]
    if cand and cand[0] != REPO:
        shutil.move(str(cand[0]), str(REPO))
    print("Đã giải nén từ tệp nén vào", REPO)
else:
    r = subprocess.run(["git", "clone", "--depth", "1", GITHUB_URL, str(REPO)],
                       capture_output=True, text=True)
    print(r.stdout[-500:])
    print(r.stderr[-500:])

assert (REPO / "train_grafiti.py").exists(), "Chưa có mã nguồn GraFITi trong thư mục Code"

if not GIU_LAI_DU_LIEU_MAU:
    for sub in ("data", ".git"):
        p = REPO / sub
        if p.exists():
            shutil.rmtree(p, ignore_errors=True)
            print("Đã xóa cho nhẹ Drive:", p)

tong = sum(f.stat().st_size for f in REPO.rglob("*") if f.is_file())
print("Dung lượng kho mã sau khi dọn:", round(tong / 1e6, 1), "MB")


9)
Updating files:  95% (12483/13139)
Updating files:  95% (12572/13139)
Updating files:  96% (12614/13139)
Updating files:  96% (12672/13139)
Updating files:  97% (12745/13139)
Updating files:  97% (12771/13139)
Updating files:  97% (12864/13139)
Updating files:  98% (12877/13139)
Updating files:  98% (12923/13139)
Updating files:  99% (13008/13139)
Updating files:  99% (13026/13139)
Updating files:  99% (13122/13139)
Updating files: 100% (13139/13139)
Updating files: 100% (13139/13139), done.

Dung lượng kho mã sau khi dọn: 1551.5 MB


## 2. Liệt kê các tệp được dùng lại

Bốn tệp đáng quan tâm nhất: hai tệp định nghĩa đường cơ sở tuyến tính, một tệp định nghĩa nhiệm vụ trên PhysioNet 2012, một tệp huấn luyện. Phần còn lại chỉ để tra cứu.

In [3]:
QUAN_TAM = [
    "baseline_experiments/FEDformer/models/DLinear.py",
    "baseline_experiments/FEDformer/models/NLinear.py",
    "baseline_experiments/train_former.py",
    "tsdm/tasks/physionet2012.py",
    "tsdm/datasets/physionet2012.py",
    "train_grafiti.py",
    "grafiti/grafiti.py",
    "grafiti/grafiti_layers.py",
]

kiem_ke = []
for rel in QUAN_TAM:
    p = REPO / rel
    if p.exists():
        so_dong = sum(1 for _ in p.open("r", errors="ignore"))
        kiem_ke.append({"tep": rel, "so_dong": so_dong, "kich_thuoc_byte": p.stat().st_size})
        print(rel.ljust(48), so_dong, "dòng")
    else:
        kiem_ke.append({"tep": rel, "so_dong": None, "kich_thuoc_byte": None})
        print(rel.ljust(48), "KHÔNG TÌM THẤY")

(OUT / "W2_grafiti_kiem_ke.json").write_text(
    json.dumps({"thoi_diem": datetime.datetime.now().isoformat(timespec="seconds"),
                "duong_dan": str(REPO), "tep": kiem_ke}, ensure_ascii=False, indent=2))
print("Đã ghi bảng kiểm kê vào", OUT / "W2_grafiti_kiem_ke.json")

baseline_experiments/FEDformer/models/DLinear.py 88 dòng
baseline_experiments/FEDformer/models/NLinear.py 38 dòng
baseline_experiments/train_former.py             332 dòng
tsdm/tasks/physionet2012.py                      348 dòng
tsdm/datasets/physionet2012.py                   476 dòng
train_grafiti.py                                 298 dòng
grafiti/grafiti.py                               129 dòng
grafiti/grafiti_layers.py                        317 dòng
Đã ghi bảng kiểm kê vào /content/drive/MyDrive/AIO2026_Conquer_M4_DLinear_NLinear_Slot4/Output/W2_grafiti_kiem_ke.json


## 3. Đối chiếu DLinear và NLinear

Hai điểm cần xác minh bằng số, không phải bằng lời.

- Trong mã của nhóm tác giả, chiều vào của lớp tuyến tính bằng hai lần độ dài cửa sổ, vì mặt nạ quan sát được nối vào sau chuỗi giá trị theo trục thời gian. Đây chính là ý nghĩa của dấu cộng trong tên DLinear+ và NLinear+.
- Trong NLinear của nhóm tác giả, giá trị neo được lấy tại hàng cuối cùng của tenxơ đã nối. Sau khi nối, hàng cuối cùng thuộc phần mặt nạ, nên neo nhận giá trị 0 hoặc 1 thay vì một giá trị sinh lý.

In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from types import SimpleNamespace

sys.path.insert(0, str(REPO / "baseline_experiments" / "FEDformer"))
from models import DLinear as DLinear_goc
from models import NLinear as NLinear_goc

L, H, C = 36, 12, 37
cfg = SimpleNamespace(seq_len=L, pred_len=H, enc_in=C, individual=False)

m_dl = DLinear_goc.Model(cfg)
m_nl = NLinear_goc.Model(cfg)
print("Chiều vào của lớp tuyến tính trong DLinear gốc:", m_dl.Linear_Trend.in_features)
print("Chiều vào của lớp tuyến tính trong NLinear gốc:", m_nl.Linear.in_features)
print("Độ dài cửa sổ khai báo:", L, "nên hệ số nhân đúng bằng", m_nl.Linear.in_features / L)

36
Chiều vào của lớp tuyến tính trong DLinear gốc: 72
Chiều vào của lớp tuyến tính trong NLinear gốc: 72
Độ dài cửa sổ khai báo: 36 nên hệ số nhân đúng bằng 2.0


In [5]:
# Tái dựng tối thiểu phần triển khai của chúng ta để so các con số.
def moving_avg_ta(x, kernel_size=25):
    front = x[:, :1, :].repeat(1, (kernel_size - 1) // 2, 1)
    end = x[:, -1:, :].repeat(1, kernel_size - (kernel_size - 1) // 2 - 1, 1)
    xp = torch.cat([front, x, end], dim=1).permute(0, 2, 1)
    return F.avg_pool1d(xp, kernel_size=kernel_size, stride=1).permute(0, 2, 1)


torch.manual_seed(0)
x = torch.randn(8, L, C)
mask = (torch.rand(8, L, C) > 0.7).float()
x = x * mask
vao_goc = torch.cat([x, mask], dim=1)   # nối theo trục thời gian, đúng như train_former.py

# DLinear: chúng ta dùng cùng bộ trọng số để kiểm tra tính tương đương.
voi_goc_dl = m_dl(vao_goc)

lin_s = nn.Linear(2 * L, H)
lin_t = nn.Linear(2 * L, H)
lin_s.load_state_dict(m_dl.Linear_Seasonal.state_dict())
lin_t.load_state_dict(m_dl.Linear_Trend.state_dict())

trend = moving_avg_ta(vao_goc)
mua_vu = vao_goc - trend
B = vao_goc.shape[0]
z = lin_s(mua_vu.permute(0, 2, 1)) + lin_t(trend.permute(0, 2, 1))
cua_ta_dl = z.permute(0, 2, 1)

chenh_dl = (voi_goc_dl - cua_ta_dl).abs().max().item()
print("Chênh lệch lớn nhất giữa hai cách tính DLinear:", chenh_dl)

Chênh lệch lớn nhất giữa hai cách tính DLinear: 0.0


In [6]:
# NLinear: kiểm tra giá trị neo mà mã gốc thực sự sử dụng.
neo_goc = vao_goc[:, -1:, :]
print("Neo của mã gốc chỉ nhận các giá trị:", sorted(set(neo_goc.reshape(-1).tolist()))[:5])
print("Tỷ lệ phần tử neo bằng 0 hoặc 1:",
      round(100 * float(((neo_goc == 0) | (neo_goc == 1)).float().mean()), 2), "phần trăm")

neo_gia_tri = x[:, -1:, :]
print("Neo theo định nghĩa NLinear nguyên bản lấy từ chuỗi giá trị, độ lệch chuẩn:",
      round(float(neo_gia_tri.std()), 4))

ket_luan = {
    "chieu_vao_dlinear_goc": int(m_dl.Linear_Trend.in_features),
    "chieu_vao_nlinear_goc": int(m_nl.Linear.in_features),
    "he_so_nhan_do_dai": m_nl.Linear.in_features / L,
    "chenh_lech_toi_da_dlinear": float(chenh_dl),
    "neo_nlinear_goc_lay_tu": "hang cuoi cua tensor da noi, tuc phan mat na",
}
(OUT / "W2_doi_chieu_duong_co_so.json").write_text(
    json.dumps(ket_luan, ensure_ascii=False, indent=2))
print(json.dumps(ket_luan, ensure_ascii=False, indent=2))

Neo của mã gốc chỉ nhận các giá trị: [0.0, 1.0]
Tỷ lệ phần tử neo bằng 0 hoặc 1: 100.0 phần trăm
Neo theo định nghĩa NLinear nguyên bản lấy từ chuỗi giá trị, độ lệch chuẩn: 0.5472
{
  "chieu_vao_dlinear_goc": 72,
  "chieu_vao_nlinear_goc": 72,
  "he_so_nhan_do_dai": 2.0,
  "chenh_lech_toi_da_dlinear": 0.0,
  "neo_nlinear_goc_lay_tu": "hang cuoi cua tensor da noi, tuc phan mat na"
}


## 4. Kết luận của sổ tay này

- Mã nguồn GraFITi đã nằm trong `Code/GraFITi` và có thể tra cứu trực tiếp từ Colab.
- Định nghĩa DLinear+ và NLinear+ đã được xác minh bằng số: mặt nạ quan sát được nối vào chuỗi giá trị, làm chiều vào tăng gấp đôi.
- Điểm neo trong NLinear+ của nhóm tác giả rơi vào phần mặt nạ. Đây là một lý do kỹ thuật giải thích vì sao NLinear+ không tốt hơn DLinear+ trong bảng kết quả đã công bố, và cũng là một khoảng trống hợp lệ cho đề tài tuần 3.
- Sổ tay tiếp theo là notebook 03.